# LangGraph - Structured Outputs

**Structured outputs** refer to outputs from LLMs (or other nodes in a LangGraph workflow) that are formatted as well-defined, machine-readable objects — such as dictionaries, lists, custom classes, or JSON objects — rather than just plain text.

For example:

* Unstructured output: `"The answer is Paris."`
* Structured output: `{"city": "Paris", "country": "France"}`

In LangGraph, the output from one node is typically used as input to another. If the output is structured, the next step knows exactly what fields/data to expect and how to access them.

We can use **Pydantic** to define the schema, and **LangChain’s** structured output features to connect it all.

In [3]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

# Set up a low-temperature language model for more deterministic outputs
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# Define a Pydantic model to enforce structured output for a football player
class FootballPlayer(BaseModel):
    """
    Represents a football player with their details.
    """
    name: str = Field(description="The name of the football player")
    position: str = Field(description="The position the footballplayer plays in")
    age: str = Field(description="The age of the football player")

# Tell the language model to produce output that matches the FootballPlayer schema
llm_structured = llm.with_structured_output(FootballPlayer)

llm_structured

RunnableBinding(bound=ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x772697b4ed80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x772697999370>, root_client=<openai.OpenAI object at 0x7726a47a5dc0>, root_async_client=<openai.AsyncOpenAI object at 0x7726979999d0>, model_name='gpt-4o-mini', temperature=0.1, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True), kwargs={'response_format': <class '__main__.FootballPlayer'>, 'ls_structured_output_f

We then define a `FootballPlayer` class using Pydantic, specifying required fields (`name`, `age`, `position`) and attaching descriptions for clarity.

The key step is `llm.with_structured_output(FootballPlayer)`, which binds the language model so that its output must adhere to the structure and constraints defined by the Player schema.

In [5]:
llm_structured.invoke("Who is the best football player in the world?")

FootballPlayer(name='Lionel Messi', position='Forward', age='36')

In [6]:
llm_structured.invoke("Who was the best football player in the history?")

FootballPlayer(name='Pelé', position='Forward', age='82')

By invoking the structured LLM with a simple question, the response is automatically parsed and validated into our defined `FootballPlayer` schema.

Behind the scenes, what LangChain sends to the LLM is essentially a type of payload.

```json
{
  "model": "gpt-4o-mini",
  "temperature": 0.1,
  "messages": [
    {
      "role": "user",
      "content": "Who is the best football player in the world?"
    }
  ],
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "FootballPlayer",
        "description": "Represents a football player with their details.",
        "parameters": {
          "type": "object",
          "properties": {
            "name": {
              "type": "string",
              "description": "The name of the football player"
            },
            "age": {
              "type": "integer",
              "description": "The age of the football player"
            },
            "position": {
              "type": "string",
              "description": "The position the football player plays in"
            }
          },
          "required": ["name", "age", "position"]
        }
      }
    }
  ],
  "tool_choice": "auto"
}
```

* It creates a function/tool spec (with a name, description, and argument schema) describing the `FootballPlayer` model.

* It sends our user message (*"Who is the best football player in the world?"*) as the `user` message.
* It attaches the tool/function with the schema generated from `FootballPlayer`.

We can also use `TypedDict`.

In [11]:
from typing import TypedDict
from langchain_openai import ChatOpenAI

# Define a TypedDict schema
class PlayerDict(TypedDict):
    """Represents a football player with their details."""
    name: str
    age: int
    position: str

# Set up the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# Instruct the LLM to output using the TypedDict schema
llm_structured = llm.with_structured_output(PlayerDict)

# Make a call
llm_structured.invoke("Who was the most skilled and dribbling football player in the history?")

{'name': 'Diego Maradona', 'age': 60, 'position': 'Attacking Midfielder'}

or

In [12]:
from typing import TypedDict, Optional
from typing_extensions import Annotated
from langchain_openai import ChatOpenAI

class PlayerDict(TypedDict, total=False):
    """
    Represents a football player with their details.
    """
    name: Annotated[str, ..., "The name of the player"] # type, default value, description
    age: Annotated[int, ..., "The age of the player"]
    position: Annotated[Optional[str], None, "The position the player plays in, if known."]  # Optional

# Set up the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# Instruct the LLM to output using the TypedDict schema
llm_structured = llm.with_structured_output(PlayerDict)

# Make a call (will use position=None by default if not provided)
llm_structured.invoke("Who was the best football player in the world in 2002?")

{'name': 'Ronaldo', 'age': 46, 'position': 'Forward'}